# Microscope sync device based on a 32-bit ARM microcontroller

In [1]:
from microsync import SyncDevice, props
from time import sleep as s

## Communication logging
You can log all serial port communication by setting the `log_file` argument when you connect to the sync device.

In [2]:
log_file = None                            # no logging
log_file = 'microsync_communication.log'    # save to file

In [3]:
sd = SyncDevice("COM7", log_file='print')  # print to terminal

2025-09-04 16:35:34.029 RX: Sync device is ready. Firmware version: 2.4.4\x0A
2025-09-04 16:35:34.058 TX: GET\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00
2025-09-04 16:35:34.066 RX: 2.4.4\x0A
2025-09-04 16:35:34.066 TX: FUN\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00
2025-09-04 16:35:34.119 RX: 525721 TGL_PIN\x0A525741 SET_PIN\x0A525765 BST__ON\x0A525833 BST_OFF\x0A525893 EN__PIN\x0A525913 DIS_PIN\x0A530301 OPE_SHU\x0A530205 CLS_SHU\x0A


Let's redirect logging to a file, otherwise it will be too much text here

In [4]:
sd = SyncDevice("COM7", log_file=log_file)

COM7 is in use. Closing and re-opening...


You can get an overview of the system status with `get_status()` method, or by just printing the synchronization device:

In [5]:
sd

SYNC DEVICE v2.4.4
-- SYSTEM STATUS --
Event queue size: 0
System counter is RUNNING
System time: 0.137809 s

## Device properties
The sync device has properties (see *globals.h* file), which could be access using `get_property()` method or directly as a class attribute. Most of 
them are read-only, while others are read/write.

In [6]:
list(props)

[<props.ro_VERSION: 0>,
 <props.ro_SYS_TIMER_STATUS: 1>,
 <props.ro_SYS_TIMER_VALUE: 2>,
 <props.ro_SYS_TIMER_OVF_COUNT: 3>,
 <props.ro_SYS_TIME_s: 4>,
 <props.ro_SYS_TIMER_PRESCALER: 5>,
 <props.rw_DFLT_PULSE_DURATION_us: 6>,
 <props.ro_WATCHDOG_TIMEOUT_ms: 7>,
 <props.ro_N_EVENTS: 8>,
 <props.rw_INTLCK_ENABLED: 9>,
 <props.rw_SELECTED_LASERS: 10>,
 <props.wo_OPEN_SHUTTERS: 11>,
 <props.wo_CLOSE_SHUTTERS: 12>,
 <props.rw_SHUTTER_DELAY_us: 13>,
 <props.rw_CAM_READOUT_us: 14>,
 <props.rw_CAM_GLOBAL_RESET: 15>]

In [7]:
print(sd.get_property(props.ro_VERSION))

# or easier
print(sd.version)

2.4.4
2.4.4


In [8]:
# What is the default pulse duration?
print(f"Default pulse duration = {sd.pulse_duration_us} us")

# Let's change it
sd.pulse_duration_us = 250

print(f"Default pulse duration = {sd.pulse_duration_us} us")

Default pulse duration = 100 us
Default pulse duration = 250 us


In [9]:
# disable interlock
sd.interlock_enabled = False
sd.interlock_enabled

False

## Creating events and sending signals
You can schedule events, such as set/reset pin, toggle pin, send pulse, etc. Each event has:
 * up to two arguments
 * a corresponding timestamp,
 * number of occurences,
 * and an interval between occurences

The microcontroller puts all events into an ordered priority queue, where they are sorted by timestamp, from the earliest to the latest. If the system timer is running, it will grab the next scheduled event and execute it exactly at the timestamp of this event. If necessary, the event will be rescheduled.

Let's schedule a couple of positive pulses ("PPL" command in the protocol). You will see signals on Arduino pins A0 and A1, as well as a list of events (method `get_events()`). The timestamp `ts` is relative to the current time point.

In [10]:
sd.pos_pulse("A1", 50000, ts=1000000, N=25, interval=70000)
sd.pos_pulse("A0", 8000, N=120, interval=50000)
sd.get_events()

[SET_PIN(A0 , 1  ) at t=        354ms. Call    119 times every         50 ms,
 SET_PIN(A0 , 0  ) at t=        362ms. Call    119 times every         50 ms,
 SET_PIN(A1 , 1  ) at t=       1302ms. Call     25 times every         70 ms,
 SET_PIN(A1 , 0  ) at t=       1352ms. Call     25 times every         70 ms]

### Context manager
In the previous example, events were scheduled using individual lines of Python code, which resulted in individual data transmission to the sync device. The instructions are processed immediately, however, they can arrive 10-25 ms apart from each other, and there is no way to predict the time delay your computer puts between them.

If you want tighter timings between the requested events, use the context manager. Every command within the context manager gets queued in a buffer, and the entire buffer is transmitted the moment you exit the context manager. The time delay between subsequent commands should be exactly 1.88ms (time to transmit 24 bytes at 115200 baud). Below is the example

**NOTE** Pin A12 is sometimes labeled "DAC2" on Arduino Due.

In [11]:
with sd as dev:
    dev.pos_pulse("A12",100000, N=10, interval=500000, ts=0)
    dev.pos_pulse("A0", 100000, N=10, interval=500000, ts=5000)
    dev.pos_pulse("A12",100000, N=10, interval=500000, ts=10000)
    dev.pos_pulse("A1", 100000, N=10, interval=500000, ts=105000)
    dev.pos_pulse("A12",100000, N=10, interval=500000, ts=200000)
    dev.pos_pulse("A2", 100000, N=10, interval=500000, ts=205000)
    dev.pos_pulse("A12",100000, N=10, interval=500000, ts=300000)
    dev.pos_pulse("A3", 100000, N=10, interval=500000, ts=305000)
s(0.5)
sd.get_events()

[SET_PIN(A0 , 1  ) at t=        954ms. Call    107 times every         50 ms,
 SET_PIN(A0 , 0  ) at t=        962ms. Call    107 times every         50 ms,
 SET_PIN(A12, 0  ) at t=        994ms. Call      9 times every        500 ms,
 SET_PIN(A0 , 0  ) at t=       1001ms. Call      9 times every        500 ms,
 SET_PIN(A1 , 1  ) at t=       1005ms. Call      9 times every        500 ms,
 SET_PIN(A12, 0  ) at t=       1008ms. Call      9 times every        500 ms,
 SET_PIN(A12, 1  ) at t=       1102ms. Call      9 times every        500 ms,
 SET_PIN(A1 , 0  ) at t=       1105ms. Call      9 times every        500 ms,
 SET_PIN(A2 , 1  ) at t=       1109ms. Call      9 times every        500 ms,
 SET_PIN(A12, 0  ) at t=       1202ms. Call      9 times every        500 ms,
 SET_PIN(A12, 1  ) at t=       1206ms. Call      9 times every        500 ms,
 SET_PIN(A2 , 0  ) at t=       1209ms. Call      9 times every        500 ms,
 SET_PIN(A3 , 1  ) at t=       1213ms. Call      9 times every  

## Precise timings with stopping/starting system timer
If you have several events and need exact time intervals between them, you should stop system timer, program your events, and then re-start the system timer. No events are processes when the timer is stopped.


In [12]:
sd.running

True

In [13]:
sd.stop()
sd.running

False

In [14]:
sd.pos_pulse("A12",100000, N=10, interval=500000, ts=0)
sd.pos_pulse("A0", 100000, N=10, interval=500000, ts=5000)
sd.pos_pulse("A12",100000, N=10, interval=500000, ts=10000)
sd.pos_pulse("A1", 100000, N=10, interval=500000, ts=105000)
sd.pos_pulse("A12",100000, N=10, interval=500000, ts=200000)
sd.pos_pulse("A2", 100000, N=10, interval=500000, ts=205000)
sd.pos_pulse("A12",100000, N=10, interval=500000, ts=300000)
sd.pos_pulse("A3", 100000, N=10, interval=500000, ts=305000)
sd.get_events()

[SET_PIN(A12, 1  ) at t=          0ms. Call     10 times every        500 ms,
 SET_PIN(A0 , 1  ) at t=          5ms. Call     10 times every        500 ms,
 SET_PIN(A12, 1  ) at t=         10ms. Call     10 times every        500 ms,
 SET_PIN(A12, 0  ) at t=        100ms. Call     10 times every        500 ms,
 SET_PIN(A1 , 1  ) at t=        105ms. Call     10 times every        500 ms,
 SET_PIN(A0 , 0  ) at t=        105ms. Call     10 times every        500 ms,
 SET_PIN(A12, 0  ) at t=        110ms. Call     10 times every        500 ms,
 SET_PIN(A12, 1  ) at t=        200ms. Call     10 times every        500 ms,
 SET_PIN(A2 , 1  ) at t=        205ms. Call     10 times every        500 ms,
 SET_PIN(A1 , 0  ) at t=        205ms. Call     10 times every        500 ms,
 SET_PIN(A12, 1  ) at t=        300ms. Call     10 times every        500 ms,
 SET_PIN(A12, 0  ) at t=        300ms. Call     10 times every        500 ms,
 SET_PIN(A3 , 1  ) at t=        305ms. Call     10 times every  

### Visualization of event order
You can call method `show_events()` to open a plot showing all scheduled events. The plot is interactive (it uses Bokeh), so you can zoom in and examine the exact order and timing of scheduled events.

In [15]:
sd.show_events()

Loading BokehJS ...

<microsync.microsync.SyncDevice.show_events.<locals>.PlotWrapper at 0x1f1c150f680>

Let's add two more events that would toggle pin D4 at close frequency, causing the connected LED to gradually change apparent brightness.

In [16]:
sd.tgl_pin("D17", N=10000, interval=500)
sd.tgl_pin("D17", N=10000, interval=501)

In [17]:
sd

SYNC DEVICE v2.4.4
-- SYSTEM STATUS --
Event queue size: 18
System counter is STOPPED
System time: 0.000000 s

In [18]:
# Now we have 18 scheduled events, let them run
sd.go()
s(0.5)
sd.get_events("us")

[TGL_PIN(D17, 0  ) at t=     514803us. Call   8970 times every        499 us,
 TGL_PIN(D17, 0  ) at t=     514979us. Call   8972 times every        500 us,
 SET_PIN(A12, 0  ) at t=     600000us. Call      9 times every     500000 us,
 SET_PIN(A1 , 1  ) at t=     605000us. Call      9 times every     500000 us,
 SET_PIN(A0 , 0  ) at t=     605000us. Call      9 times every     500000 us,
 SET_PIN(A12, 0  ) at t=     610000us. Call      9 times every     500000 us,
 SET_PIN(A12, 1  ) at t=     700000us. Call      9 times every     500000 us,
 SET_PIN(A1 , 0  ) at t=     705000us. Call      9 times every     500000 us,
 SET_PIN(A2 , 1  ) at t=     705000us. Call      9 times every     500000 us,
 SET_PIN(A12, 1  ) at t=     800000us. Call      9 times every     500000 us,
 SET_PIN(A12, 0  ) at t=     800000us. Call      9 times every     500000 us,
 SET_PIN(A2 , 0  ) at t=     805000us. Call      9 times every     500000 us,
 SET_PIN(A3 , 1  ) at t=     805000us. Call      9 times every  

If you connect oscilloscope, you will see that the time intervals between the events are exactly what you requested, down to microseconds. You will see jitter is two or more events occur very close to each other (closer than ~10us). If two events are scheduled to happen at the same timestamp, their order is undefined. Avoid scheduling of same events if you can.

#### Watchdog timer and crash detection
Let's try to schedule way too many events, also schedule them with the same timestamp, and see what happens.

In [19]:
sd.stop()
for i in range(400):
    sd.tgl_pin("D17", ts=25*i, N=10000, interval=1000)
len(sd.get_events())

400

Now we have too many events in the table. Once we start the microcontroller, it will slowly get overwhelmed and will start missing events. When this happens, it will automatically reset. It will send an error message to the host.

In [20]:
sd.go()
s(1)
print(sd.com.readall().decode())

ERR - watchdog timeout; restarting system!
 Sync device is ready. Firmware version: 2.4.4



## Enabling/disabling pins
Each pin can be enabled or disabled using the "ENP" and "DSP" commands. A disabled pin still preserves internal state, but it lacks the electrical output.

In [21]:
with sd as dev:
    dev.tgl_pin("D17", N=5000, interval=1000)
    dev.tgl_pin("D17", N=5000, interval=1001)
    dev.tgl_pin("A0", N=5000, interval=1000)
    dev.tgl_pin("A0", N=5000, interval=1001)
s(0.5)
sd.disable_pin("D17")
s(0.5)
sd.enable_pin("D17")

## Special pins — laser shutter control
Some pins (see *globals.h*) are assigned to laser shutters. These have special behavior.

### Interlock
Laser shutter pins are monitored by an interlock for laser safety purposes. The interlock periodically sends a short heart beat pulse on pin D13 and expects to receive this pulse on pin D12. If the pulse does not arrive, the laser shutter pins are immediately set to TTL low. The pins return to their normal function as soon as interlock detects pulse on D12. This mechanism does not affect the enable/disable status of the pins or their internal level, it's a separate layer of logic.

Sending hearbeat pulse through the interlock circuit makes the interlock highly reliable, as it detects not only the open circuit, but also short circuit to ground or TTL high.

If you don't need interlock, you can disable it until the next microcontroller reset.

In [22]:
sd.interlock_enabled

True

In [23]:
# Interlock is disabled, A0 is unaffected
sd.clear()
sd.interlock_enabled = False
sd.tgl_pin("A0", N=10000, interval=1000)
sd.tgl_pin("A0", N=10000, interval=1001)
sd.tgl_pin("A12", N=10000, interval=1000)
sd.tgl_pin("A12", N=10000, interval=1001)
s(1)

In [24]:
# A0 is disabled if interlock circuit is disrupted
sd.clear()
sd.interlock_enabled = True
sd.tgl_pin("A0", N=10000, interval=1000)
sd.tgl_pin("A0", N=10000, interval=1001)
sd.tgl_pin("A12", N=10000, interval=1000)
sd.tgl_pin("A12", N=10000, interval=1001)
s(1)

### Shortcuts to control laser shutters
There are device properties that you can use to open/close laser shutters, as wells as enable/disable specific lasers. All of them accept bitmask indicating which spectral channels are affected, with bit 0 corresponding to Cy2, and bit 3 corresponding to Cy7.

The methods `open_shutters` and `close_shutters` set the internal level state of the corresponding pin (analogous to `PIN` command).

In [25]:
sd.interlock_enabled = False
sd.stop()

In [26]:
# open all laser shutters immediately, even if the system timer is not running
sd.open_shutters()

In [27]:
# close Cy5 shutter (bit 2)
sd.close_shutters(1 << 2)

In [28]:
# Open Cy3 and Cy5 shutter
sd.open_shutters(1 << 1 | 1 << 2)

In [29]:
# close all shutters
sd.close_shutters()

The `selected_lasers` property accepts a bitmask indicating what laser channels are in use. It enables/disables output of the corresponding channel, analogous to `ENP` and `DSP` commands.

In [30]:
sd.stop()
sd.pos_pulse("A0", 50000, interval=200000)
sd.pos_pulse("A1", 50000, interval=205000)
sd.pos_pulse("A2", 50000, interval=210000)
sd.pos_pulse("A3", 50000, interval=215000)
sd.show_events()
sd.go()
sd.selected_lasers = 0b0010 # only Cy3

Loading BokehJS ...

In [31]:
sd.selected_lasers = 0 # none of them
s(1)
sd.selected_lasers = 0b0110  # only Cy3 and Cy5
s(1)
sd.selected_lasers = 0b1111 # all of them

## Shortcuts to pTIRF acquisition modes
The commands `CON`, `STR`, and `ALX` are shortcuts to continuous, stroboscopic, and ALEX data acquisition modes.

In [32]:
# Specify system settings
sd.cam_readout_us = 12_000
sd.shutter_delay_us = 1_500
sd.pulse_duration_us = 200

### Continuous acquisition
In this mode, each camera frame ends on the rising edge of the TTL signal on pin A12, and immediately the next frame exposure begins. The camera runs in the rolling shutter mode, with lines being subsequently read out. It takes `sd.cam_readout_us` microseconds to read out the entire chip.

We begin with a sacrificial frame (frame 0). We open laser shutters during the sacrificial frame, and then give N+1 pulses to the camera, and start closing the laser shutters at the end of the last frame. We assume that it takes `sd.shutter_delay_us` microseconds to open or close laser shutters, which is indicated with the pale wheat color on the graph.

Note that the first frame is scheduled to happen exactly at the requested timestamp, i.e. the lasers shutters and sacrificial frame are happening in advance, if possible.

In [33]:
sd.stop()
sd.selected_lasers = 0b0110
# Start continuous data acquisition with Cy3 and Cy5 lasers
sd.start_continuous_acq(exp_time=20_000, ts=50_000, N_frames=7)
p = sd.show_events()
sd.go()
s(1)
p.save_plot("continuous_acq.svg")

Loading BokehJS ...

Successfully saved SVG to continuous_acq.svg


Let's see how the system handles extremely fast imaging, when the camera exposure is short compared to the shutter transition time. On top of that, we have another digital pulse on pin D5 for some third-party equipment.

In [34]:
sd.stop()
# Continuous acquisition with very fast time resolution
sd.start_continuous_acq(exp_time=200, ts=15_000, N_frames=12)
sd.pos_pulse("D5", duration=1_000, ts=12_000)
sd.show_events()
sd.go()
s(1)

Loading BokehJS ...

### Stroboscopic imaging
In this mode, the camera starts collecting photons, and we briefly open the laser shutters for a short moment. After that, we have a period of darkness that we use to read out the camera. This way we can take a full-resolution snapshot with very short exposure time, but, the time is not continuous anymore.

In [35]:
sd.stop()
sd.selected_lasers = 0b1011
# Start stroboscopic data acquisition with Cy2, Cy3 and Cy7 lasers
sd.start_stroboscopic_acq(exp_time=5_000, N_frames=6)
p = sd.show_events()
sd.go()
s(1)
p.save_plot("stroboscopic_acq.svg")

Loading BokehJS ...

Successfully saved SVG to stroboscopic_acq.svg


In this mode, we can also do time lapse imaging by specifying the `frame_period`.

In [36]:
sd.stop()
sd.selected_lasers = 0b1001
# Start stroboscopic timelapse data acquisition with Cy2 and Cy7 lasers
sd.start_stroboscopic_acq(exp_time=200_000, N_frames=5, frame_period=1500000)
sd.show_events()
sd.go()
s(1)

Loading BokehJS ...

### Alternating laser excitation (ALEX) imaging
This mode is similar to stroboscopic. However, with each subsequent frame, we illuminate the sample with a different laser.

In [37]:
sd.stop()
sd.selected_lasers = 0b1101
# Start ALEX data acquisition with three lasers
sd.start_ALEX_acq(exp_time=10_000, N_bursts=4)
sd.show_events()
sd.go()
s(1)

Loading BokehJS ...

ALEX imaging can also be combined with time lapse, if you provide a sufficiently large `burst_period` argument.

In [38]:
sd.stop()
sd.selected_lasers = 0b1111
# Start ALEX timelapse data acquisition with all four lasers
sd.start_ALEX_acq(exp_time=15_000, N_bursts=6, burst_period=150_000)
p = sd.show_events()
sd.go()
s(1)
p.save_plot("ALEX_acq.svg")

Loading BokehJS ...

Successfully saved SVG to ALEX_acq.svg


## Hamamatsu vs Kinetix level trigger mode
We use Hamamatsu cameras in the special mode called *Global Reset level trigger*. When the camera detects the trigger, it immediately clears the entire sensor and starts exposing it. Therefore, we open the laser shutter a moment before triggering the camera. After we drop the camera level trigger to LOW, we have to wait for the camera to read out the sensor before we can send the next pulse.

This special behavior is activated when you set property `rw_CAM_GLOBAL_RESET` to 1. Compare the timing diagram below to the one above.

In [39]:
sd.stop()
sd.selected_lasers = 0b1111
sd.cam_global_rst = True;

# Start ALEX timelapse data acquisition with all four lasers
sd.start_ALEX_acq(exp_time=15_000, N_bursts=6, burst_period=150_000)
p = sd.show_events()
sd.go()
s(1)
p.save_plot("ALEX_acq_global_reset.svg")

Loading BokehJS ...

Successfully saved SVG to ALEX_acq_global_reset.svg


If you are using any of the pTIRF acquisition modes to setup data acquisition, you can check how many frames are left

In [40]:
sd.N_frames_left()

0